## Cell 1: Imports and Setup

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import json
import os
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import time

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Check GPU availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Cell 2: Load Dataset Info

In [ ]:
# NOTE: Verify the path matches where Kaggle mounted the dataset. 
# Usually it is /kaggle/input/<dataset-slug>/
JSON_PATH = "data/dataset/brain-tumor-mri-images-30-classes/DATA.json"
IMG_DIR = "data/dataset/brain-tumor-mri-images-30-classes"
OUTPUT_MODEL_CKPTS = "ckpts/dinov2_brain_tumor/best_dinov2_brain_tumor.pth

os.makedirs(os.path.dirname(OUTPUT_MODEL_CKPTS), exist_ok=True)

with open(JSON_PATH, 'r', encoding='utf-8') as f:
    data = json.load(f)

dataset_info = []
classes_set = set()

for rel_path, info in data.items():
    # Handle path separators
    safe_rel_path = rel_path.replace('\\', os.sep).replace('/', os.sep)
    full_img_path = os.path.join(IMG_DIR, safe_rel_path)
    
    if not os.path.exists(full_img_path):
        continue
        
    class_name = info['class']
    classes_set.add(class_name)
    dataset_info.append({
        'img_path': full_img_path,
        'class': class_name
    })

# Sort classes to ensure consistent indexing
classes_list = sorted(list(classes_set))
class_to_idx = {cls: idx for idx, cls in enumerate(classes_list)}
num_classes = len(classes_list)

print(f"Total images loaded: {len(dataset_info)}")
print(f"Number of classes identified: {num_classes}")

## Cell 3: Define PyTorch Dataset and Transforms

In [ ]:
class BrainTumorDataset(Dataset):
    def __init__(self, dataset_info, class_to_idx, transform=None):
        self.dataset_info = dataset_info
        self.class_to_idx = class_to_idx
        self.transform = transform

    def __len__(self):
        return len(self.dataset_info)

    def __getitem__(self, idx):
        img_path = self.dataset_info[idx]['img_path']
        class_name = self.dataset_info[idx]['class']
        label = self.class_to_idx[class_name]
        
        # Open and convert image to RGB
        img = Image.open(img_path).convert('RGB')
        
        if self.transform:
            img = self.transform(img)
            
        return img, label

IMG_SIZE = 518 

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomApply([transforms.ColorJitter(brightness=0.15, contrast=0.15)], p=0.4),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05), scale=(0.95, 1.05)),
    transforms.ToTensor(),
    # Standard ImageNet normalization used by DINOv2
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Train/Validation Split (Stratified to keep class distribution balanced)
train_info, val_info = train_test_split(
    dataset_info, 
    test_size=0.2, 
    random_state=42, 
    stratify=[d['class'] for d in dataset_info]
)

train_dataset = BrainTumorDataset(train_info, class_to_idx, train_transform)
val_dataset = BrainTumorDataset(val_info, class_to_idx, val_transform)

# Batch size 16 is safe for 16GB GPUs (like Tesla T4 or P100) with 518x518 images
BATCH_SIZE = 16 
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

## Cell 4: Define DINOv2 Model and Classification Head

In [ ]:
print("Loading DINOv2 model (this may take a moment to download weights)...")
# 'dinov2_vitb14' is a great balance of speed and accuracy.
backbone = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitb14')

# Freeze the backbone parameters to save memory and speed up training.
for param in backbone.parameters():
    param.requires_grad = False

class DINOv2Classifier(nn.Module):
    def __init__(self, backbone, num_classes, hidden_dim=512):
        super(DINOv2Classifier, self).__init__()
        self.backbone = backbone
        self.embed_dim = backbone.embed_dim # 768 for ViT-B/14
        
        # Define a simple MLP classification head
        self.classifier = nn.Sequential(
            nn.Linear(self.embed_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(0.4),
            nn.Linear(hidden_dim, num_classes)
        )
        
    def forward(self, x):
        # Extract features from DINOv2 and get the normalized CLS token
        features = self.backbone.forward_features(x)["x_norm_clstoken"]
        out = self.classifier(features)
        return out

model = DINOv2Classifier(backbone, num_classes).to(device)
print(f"Model initialized on {device}")

## Cell 5: Training Loop

In [ ]:
criterion = nn.CrossEntropyLoss()

# We only optimize the parameters of the classification head
optimizer = optim.AdamW(model.classifier.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2, verbose=True)

EPOCHS = 15
best_val_acc = 0.0

print("Starting training...")
for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    correct_train = 0
    total_train = 0
    
    start_time = time.time()
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()
        
    train_acc = correct_train / total_train
    train_loss = train_loss / total_train
    
    # Validation phase
    model.eval()
    val_loss = 0.0
    correct_val = 0
    total_val = 0
    
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()
            
    val_acc = correct_val / total_val
    val_loss = val_loss / total_val
    
    scheduler.step(val_acc)
    
    epoch_time = time.time() - start_time
    print(f"Epoch [{epoch+1}/{EPOCHS}] ({epoch_time:.1f}s) - "
          f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")
          
    # Save the best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), OUTPUT_MODEL_CKPTS)
        print(f"-> Best model saved with Val Acc: {best_val_acc:.4f}")

print("Training finished!")

## Cell 6: Evaluation and Visualization

In [ ]:
# Load the best weights
model.load_state_dict(torch.load(OUTPUT_MODEL_CKPTS, map_location=device))
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())

# Print Classification Report
print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=classes_list))

# Plot Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(20, 16))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes_list, yticklabels=classes_list)
plt.title('Confusion Matrix - DINOv2')
plt.ylabel('Real Class')
plt.xlabel('Predicted Class')
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('confusion_matrix_dinov2.png')
plt.show()